# Binary Classification Model

Task: Develop the best predictive model you can for (a) binary classification. You can use any preprocessing and any modeling technique you like, including ones not discussed in class (consider exploring RNN or SVM). Export a deployable copy of each model and a set of predictions from each model on the test data claims-test.RData. Predictions should be formatted as a single data frame named pred_df with columns

- .id containing the URL ID

- bclass.pred containing the predicted label for the binary class setting

- mclass.pred containing the predicted label for the multiclass setting


In [33]:
import pyreadr
import re
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## Load Data

In [15]:
raw_data_df = pyreadr.read_r('../../data/claims-raw.RData')["claims_raw"]
test_data_df = pyreadr.read_r('../../data/claims-test.RData')["claims_test"]


## Preprocess Claims Data 

Sourced from preprocessing.R

In [18]:
# function to parse html and clean text 
def parse_fn(html, selector = None):
    soup = BeautifulSoup(html, features = "html.parser")
    try:
        for script in soup(["script", "style"]):
            script.decompose() # destroy script and style elements
        if selector is None:
            raw = ' '.join(spoon.get_text(' '))
        else:
            raw = ' '.join(spoon.get_text(' ') for spoon in soup.select(selector))
    
        text = re.sub(r'\S+@\S+|https?://\S+|www\.\S+', "", raw)
        text = re.sub("'", "",text)
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
        test = re.sub(r'nbsp', ' ', text)
        text = text.lower()
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text 
    
    except Exception as err:
        print(f"Error parsing HTML: {err}")
        return None

# function to apply to test claims data
def parse_data(df, selector = None):
    df = df[df['text_tmp'].str.contains('<!', regex = False)].copy() 
    df['text_clean'] = df['text_tmp'].apply(parse_fn, selector = selector)
    return df

In [19]:
clean_claims_df = parse_data(raw_data_df, selector = 'p')
test_clean_claims_df = parse_data(test_data_df, selector = 'p')

### Partition 

In [22]:
clean_claims_df.head()

,neo_search_transaction_id,neo_search_subject_id,original_url,text_tmp,internal_feedback,.id,label,mclass,bclass,text_clean
0,12395162.0,11497914.0,http://hosting-22647.tributes.com/obituary/sho...,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 S...",Possible Fatality,url2183,Possible Fatality,Possible Fatality,Relevant claim content,national obituary search click on the item you...
1,12394582.0,11499214.0,https://www.localcrimenews.com/welcome/arrest/...,"<!DOCTYPE html>\n<html lang=""en"">\n<head>\n ...",Potentially unlawful activity,url2136,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,the following official arrest record for jonat...
2,12384260.0,11485861.0,https://www.bustedmugshots.com/florida/miami-d...,<!DOCTYPE html>\n<html>\n<head>\n<meta charset...,N/A: No relevant content.,url1586,N/A: No relevant content.,N/A: No relevant content.,N/A: No relevant content.,did someone you know get arrested in miami dad...
3,12363093.0,11481705.0,https://www.policearrests.com/florida-arrest-r...,"<!DOCTYPE HTML>\n<html lang=""en"">\n<head>\n\t<...",Potentially unlawful activity,url209,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,the information on this website is taken from ...
4,12376392.0,11488536.0,https://www.bustedmugshots.com/tennessee/memph...,<!DOCTYPE html>\n<html>\n<head>\n<meta charset...,Potentially unlawful activity,url628,Potentially unlawful activity,Potentially unlawful activity,Relevant claim content,name clayton thomas location memphis tennessee...


In [28]:
data = clean_claims_df.dropna(subset=['text_clean'])
data = data[data['text_clean'].str.len() > 0] # drop rows with empty string 

targets, class_names = pd.factorize(data['bclass']) # convert labels to 0, 1 

X_train_text, X_val_text, y_train, y_val = train_test_split(data['text_clean'], targets, test_size = 0.2, stratify = targets, random_state = 2026)

### Vectorize

In [31]:
vectorizer = TfidfVectorizer(stop_words = 'english', min_df = 2, max_features = 10000)
X_train = vectorizer.fit_transform(X_train_text).toarray().astype('float32') # fit - learn vocab and IDF for training set 
X_val = vectorizer.transform(X_val_text).toarray().astype('float32') # transform only 

X_train.shape

(1575, 10000)

### Model 

Test Different Weights 

Train with Best Weights

Predict with Test Set 